In [4]:
import pandas as pd
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Configuration
CSV_FILE = "improved_predictions.csv"
FITS_FOLDER = 'test'
OUTPUT_BASE = 'confusion_matrix_galaxies_rf5'
GALAXIES_PER_IMAGE = 4

def normalize_galaxy_name(name):
    """Convert galaxy name to match FITS filename format"""
    # Replace spaces with underscores and handle special characters
    normalized = name.replace(' ', '_').replace('-', '-')
    return normalized

def find_fits_file(galaxy_name, fits_folder):
    """Find the corresponding FITS file for a galaxy"""
    normalized_name = normalize_galaxy_name(galaxy_name)
    
    # Possible filename patterns
    patterns = [
        f"norm_resized_{normalized_name}.fits",
        f"norm_resized_{normalized_name.replace('_', '-')}.fits",
        f"norm_resized_{normalized_name.replace('-', '_')}.fits",
    ]
    
    for pattern in patterns:
        filepath = Path(fits_folder) / pattern
        if filepath.exists():
            return str(filepath)
    
    return None

def load_fits_image(fits_path):
    """Load and return image data from FITS file"""
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data
            # Handle different data shapes
            if data.ndim > 2:
                data = data[0] if data.shape[0] < data.shape[-1] else data
            return data
    except Exception as e:
        print(f"Error loading {fits_path}: {e}")
        return None

def create_galaxy_grid(galaxy_data_list, output_path, category_name):
    """Create a 2x2 grid of galaxy images"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(category_name, fontsize=16, fontweight='bold')
    
    axes = axes.flatten()
    
    for idx, (name, data, pred_label, true_label, prob) in enumerate(galaxy_data_list):
        if idx >= 4:
            break
            
        ax = axes[idx]
        
        if data is not None:
            # Display image with appropriate scaling
            vmin, vmax = np.percentile(data, [1, 99])
            im = ax.imshow(data, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
            
            # Add colorbar
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
            # Add title with galaxy info
            pred_str = "Barred" if pred_label == 1 else "Unbarred"
            true_str = "Barred" if true_label == 1 else "Unbarred"
            title = f"{name}\nTrue: {true_str}, Pred: {pred_str}\nConf: {prob:.3f}"
            ax.set_title(title, fontsize=10, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'{name}\nNot Found', 
                   ha='center', va='center', fontsize=12)
            ax.set_title(f"{name} (Missing)", fontsize=10)
        
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(len(galaxy_data_list), 4):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

def process_galaxies():
    """Main processing function"""
    # Load CSV
    print("Loading predictions CSV...")
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded {len(df)} galaxies")
    
    # Create output directories
    categories = {
        'TN': ('True_Negative_Unbarred_Unbarred', 0, 0),  # True=0, Pred=0
        'FP': ('False_Positive_Unbarred_Barred', 0, 1),    # True=0, Pred=1
        'FN': ('False_Negative_Barred_Unbarred', 1, 0),    # True=1, Pred=0
        'TP': ('True_Positive_Barred_Barred', 1, 1)        # True=1, Pred=1
    }
    
    # Create base output directory
    base_path = Path(OUTPUT_BASE)
    base_path.mkdir(exist_ok=True)
    
    for cat_key, (folder_name, true_val, pred_val) in categories.items():
        cat_path = base_path / folder_name
        cat_path.mkdir(exist_ok=True)
        
        # Filter galaxies for this category
        mask = (df['True_Label'] == true_val) & (df['Predicted_Label'] == pred_val)
        category_galaxies = df[mask].copy()
        
        print(f"\n{'='*60}")
        print(f"Processing: {folder_name}")
        print(f"Count: {len(category_galaxies)} galaxies")
        print(f"{'='*60}")
        
        if len(category_galaxies) == 0:
            print(f"No galaxies in this category. Skipping...")
            continue
        
        # Collect galaxy data
        galaxy_data_batch = []
        batch_counter = 0
        
        for idx, row in category_galaxies.iterrows():
            galaxy_name = row['Target_Name']
            fits_path = find_fits_file(galaxy_name, FITS_FOLDER)
            
            if fits_path:
                print(f"  Found: {galaxy_name} -> {Path(fits_path).name}")
                data = load_fits_image(fits_path)
                galaxy_data_batch.append((
                    galaxy_name,
                    data,
                    row['Predicted_Label'],
                    row['True_Label'],
                    row['Confidence']
                ))
            else:
                print(f"  Missing: {galaxy_name}")
                galaxy_data_batch.append((
                    galaxy_name,
                    None,
                    row['Predicted_Label'],
                    row['True_Label'],
                    row['Confidence']
                ))
            
            # Create image when we have 4 galaxies
            if len(galaxy_data_batch) == GALAXIES_PER_IMAGE:
                batch_counter += 1
                output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
                create_galaxy_grid(galaxy_data_batch, output_file, folder_name.replace('_', ' '))
                galaxy_data_batch = []
        
        # Handle remaining galaxies
        if galaxy_data_batch:
            batch_counter += 1
            output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
            create_galaxy_grid(galaxy_data_batch, output_file, folder_name.replace('_', ' '))
    
    print("\n" + "="*60)
    print("Processing Complete!")
    print("="*60)
    print(f"\nOutput directory: {OUTPUT_BASE}/")
    print("\nFolder structure:")
    for cat_key, (folder_name, _, _) in categories.items():
        cat_path = base_path / folder_name
        if cat_path.exists():
            num_images = len(list(cat_path.glob('*.png')))
            print(f"  {folder_name}/: {num_images} images")

if __name__ == "__main__":
    process_galaxies()

Loading predictions CSV...
Loaded 117 galaxies

Processing: True_Negative_Unbarred_Unbarred
Count: 33 galaxies
  Found: UGC 09382 -> norm_resized_UGC_09382.fits
  Found: UGC 07089 -> norm_resized_UGC_07089.fits
  Found: IC 1071 -> norm_resized_IC_1071.fits
  Found: UGC 09463 -> norm_resized_UGC_09463.fits
Saved: confusion_matrix_galaxies_rf5\True_Negative_Unbarred_Unbarred\TN_batch_001.png
  Found: UGC 10468 -> norm_resized_UGC_10468.fits
  Found: NGC 4797 -> norm_resized_NGC_4797.fits
  Found: ESO 469-G015 -> norm_resized_ESO_469-G015.fits
  Found: NGC 5972 -> norm_resized_NGC_5972.fits
Saved: confusion_matrix_galaxies_rf5\True_Negative_Unbarred_Unbarred\TN_batch_002.png
  Found: ESO 291-G009 -> norm_resized_ESO_291-G009.fits
  Found: UGC 01408 -> norm_resized_UGC_01408.fits
  Found: UGC 10713 -> norm_resized_UGC_10713.fits
  Found: NGC 1380A -> norm_resized_NGC_1380A.fits
Saved: confusion_matrix_galaxies_rf5\True_Negative_Unbarred_Unbarred\TN_batch_003.png
  Found: IC 4595 -> norm_re

In [11]:
import pandas as pd
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Configuration
CSV_FILE = "rf_predictions_enhanced.csv"
FITS_FOLDER = 'test'
OUTPUT_BASE = 'confusion_matrix_galaxies_rf3'
GALAXIES_PER_IMAGE = 4

def normalize_galaxy_name(name):
    """Convert galaxy name to match FITS filename format"""
    # Replace spaces with underscores and handle special characters
    normalized = name.replace(' ', '_').replace('-', '-')
    return normalized

def find_fits_file(galaxy_name, fits_folder):
    """Find the corresponding FITS file for a galaxy"""
    normalized_name = normalize_galaxy_name(galaxy_name)
    
    # Possible filename patterns
    patterns = [
        f"norm_resized_{normalized_name}.fits",
        f"norm_resized_{normalized_name.replace('_', '-')}.fits",
        f"norm_resized_{normalized_name.replace('-', '_')}.fits",
    ]
    
    for pattern in patterns:
        filepath = Path(fits_folder) / pattern
        if filepath.exists():
            return str(filepath)
    
    return None

def load_fits_image(fits_path):
    """Load and return image data from FITS file"""
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data
            # Handle different data shapes
            if data.ndim > 2:
                data = data[0] if data.shape[0] < data.shape[-1] else data
            return data
    except Exception as e:
        print(f"Error loading {fits_path}: {e}")
        return None

def create_galaxy_grid(galaxy_data_list, output_path, category_name):
    """Create a 2x2 grid of galaxy images"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(category_name, fontsize=16, fontweight='bold')
    
    axes = axes.flatten()
    
    for idx, (name, data, pred_label, true_label, prob) in enumerate(galaxy_data_list):
        if idx >= 4:
            break
            
        ax = axes[idx]
        
        if data is not None:
            # Display image with appropriate scaling
            vmin, vmax = np.percentile(data, [1, 99])
            im = ax.imshow(data, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
            
            # Add colorbar
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
            # Add title with galaxy info
            pred_str = "Barred" if pred_label == 1 else "Unbarred"
            true_str = "Barred" if true_label == 1 else "Unbarred"
            title = f"{name}\nTrue: {true_str}, Pred: {pred_str}\nConf: {prob:.3f}"
            ax.set_title(title, fontsize=10, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'{name}\nNot Found', 
                   ha='center', va='center', fontsize=12)
            ax.set_title(f"{name} (Missing)", fontsize=10)
        
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(len(galaxy_data_list), 4):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

def process_galaxies():
    """Main processing function"""
    # Load CSV
    print("Loading predictions CSV...")
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded {len(df)} galaxies")
    
    # Check which confidence column exists and standardize
    if 'Confidence' not in df.columns and 'Prediction_Probability' in df.columns:
        df['Confidence'] = df['Prediction_Probability']
        print("Using 'Prediction_Probability' as confidence measure")
    elif 'Confidence' in df.columns:
        print("Using 'Confidence' column")
    else:
        print("Warning: No confidence measure found, using 0.5 as default")
        df['Confidence'] = 0.5
    
    # Create output directories
    categories = {
        'TN': ('True_Negative_Unbarred_Unbarred', 0, 0),  # True=0, Pred=0
        'FP': ('False_Positive_Unbarred_Barred', 0, 1),    # True=0, Pred=1
        'FN': ('False_Negative_Barred_Unbarred', 1, 0),    # True=1, Pred=0
        'TP': ('True_Positive_Barred_Barred', 1, 1)        # True=1, Pred=1
    }
    
    # Create base output directory
    base_path = Path(OUTPUT_BASE)
    base_path.mkdir(exist_ok=True)
    
    for cat_key, (folder_name, true_val, pred_val) in categories.items():
        cat_path = base_path / folder_name
        cat_path.mkdir(exist_ok=True)
        
        # Filter galaxies for this category
        mask = (df['True_Label'] == true_val) & (df['Predicted_Label'] == pred_val)
        category_galaxies = df[mask].copy()
        
        print(f"\n{'='*60}")
        print(f"Processing: {folder_name}")
        print(f"Count: {len(category_galaxies)} galaxies")
        print(f"{'='*60}")
        
        if len(category_galaxies) == 0:
            print(f"No galaxies in this category. Skipping...")
            continue
        
        # Collect galaxy data
        galaxy_data_batch = []
        batch_counter = 0
        
        for idx, row in category_galaxies.iterrows():
            galaxy_name = row['Target_Name']
            fits_path = find_fits_file(galaxy_name, FITS_FOLDER)
            
            if fits_path:
                print(f"  Found: {galaxy_name} -> {Path(fits_path).name}")
                data = load_fits_image(fits_path)
                galaxy_data_batch.append((
                    galaxy_name,
                    data,
                    row['Predicted_Label'],
                    row['True_Label'],
                    row['Confidence']
                ))
            else:
                print(f"  Missing: {galaxy_name}")
                galaxy_data_batch.append((
                    galaxy_name,
                    None,
                    row['Predicted_Label'],
                    row['True_Label'],
                    row['Confidence']
                ))
            
            # Create image when we have 4 galaxies
            if len(galaxy_data_batch) == GALAXIES_PER_IMAGE:
                batch_counter += 1
                output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
                create_galaxy_grid(galaxy_data_batch, output_file, folder_name.replace('_', ' '))
                galaxy_data_batch = []
        
        # Handle remaining galaxies
        if galaxy_data_batch:
            batch_counter += 1
            output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
            create_galaxy_grid(galaxy_data_batch, output_file, folder_name.replace('_', ' '))
    
    print("\n" + "="*60)
    print("Processing Complete!")
    print("="*60)
    print(f"\nOutput directory: {OUTPUT_BASE}/")
    print("\nFolder structure:")
    for cat_key, (folder_name, _, _) in categories.items():
        cat_path = base_path / folder_name
        if cat_path.exists():
            num_images = len(list(cat_path.glob('*.png')))
            print(f"  {folder_name}/: {num_images} images")

if __name__ == "__main__":
    process_galaxies()

Loading predictions CSV...
Loaded 117 galaxies


KeyError: 'Predicted_Label'

In [2]:
import pandas as pd
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Configuration
CSV_FILE = 'rf_predictions.csv'  # or 'rf_predictions.csv'
FITS_FOLDER = 'test'
OUTPUT_BASE = 'confusion_matrix_galaxies_rf1'
GALAXIES_PER_IMAGE = 4

def normalize_galaxy_name(name):
    """Convert galaxy name to match FITS filename format"""
    # Replace spaces with underscores and handle special characters
    normalized = name.replace(' ', '_').replace('-', '-')
    return normalized

def find_fits_file(galaxy_name, fits_folder):
    """Find the corresponding FITS file for a galaxy"""
    normalized_name = normalize_galaxy_name(galaxy_name)
    
    # Possible filename patterns
    patterns = [
        f"norm_resized_{normalized_name}.fits",
        f"norm_resized_{normalized_name.replace('_', '-')}.fits",
        f"norm_resized_{normalized_name.replace('-', '_')}.fits",
    ]
    
    for pattern in patterns:
        filepath = Path(fits_folder) / pattern
        if filepath.exists():
            return str(filepath)
    
    return None

def load_fits_image(fits_path):
    """Load and return image data from FITS file"""
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data
            # Handle different data shapes
            if data.ndim > 2:
                data = data[0] if data.shape[0] < data.shape[-1] else data
            return data
    except Exception as e:
        print(f"Error loading {fits_path}: {e}")
        return None

def detect_csv_format(df):
    """Detect which CSV format is being used and return column mappings"""
    
    # Check for column patterns
    has_rf_prefix = 'RF_Predicted_Label' in df.columns
    has_prediction_probability = 'Prediction_Probability' in df.columns
    has_confidence = 'Confidence' in df.columns
    has_rf_confidence = 'RF_Confidence' in df.columns
    has_rf_probability = 'RF_Probability_Class_1' in df.columns
    
    # Determine format and create mapping
    if has_rf_prefix:
        # Original format (rf_predictions.csv)
        column_map = {
            'predicted': 'RF_Predicted_Label',
            'true': 'True_Label',
            'confidence': 'RF_Confidence' if has_rf_confidence else 'RF_Probability_Class_1'
        }
        csv_format = "rf_predictions.csv format"
    else:
        # Enhanced format (optimized_rf_predictions_v2.csv)
        column_map = {
            'predicted': 'Predicted_Label',
            'true': 'True_Label',
            'confidence': 'Confidence' if has_confidence else 'Prediction_Probability'
        }
        csv_format = "optimized_rf_predictions_v2.csv format"
    
    return column_map, csv_format

def create_galaxy_grid(galaxy_data_list, output_path, category_name):
    """Create a 2x2 grid of galaxy images"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(category_name, fontsize=16, fontweight='bold')
    
    axes = axes.flatten()
    
    for idx, (name, data, pred_label, true_label, prob) in enumerate(galaxy_data_list):
        if idx >= 4:
            break
            
        ax = axes[idx]
        
        if data is not None:
            # Display image with appropriate scaling
            vmin, vmax = np.percentile(data, [1, 99])
            im = ax.imshow(data, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
            
            # Add colorbar
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
            # Add title with galaxy info
            pred_str = "Barred" if pred_label == 1 else "Unbarred"
            true_str = "Barred" if true_label == 1 else "Unbarred"
            title = f"{name}\nTrue: {true_str}, Pred: {pred_str}\nConf: {prob:.3f}"
            ax.set_title(title, fontsize=10, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'{name}\nNot Found', 
                   ha='center', va='center', fontsize=12)
            ax.set_title(f"{name} (Missing)", fontsize=10)
        
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(len(galaxy_data_list), 4):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

def process_galaxies():
    """Main processing function"""
    # Load CSV
    print("Loading predictions CSV...")
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded {len(df)} galaxies")
    
    # Detect CSV format and get column mappings
    column_map, csv_format = detect_csv_format(df)
    print(f"\nDetected CSV format: {csv_format}")
    print(f"Column mappings:")
    print(f"  - Predicted Label: {column_map['predicted']}")
    print(f"  - True Label: {column_map['true']}")
    print(f"  - Confidence: {column_map['confidence']}")
    
    # Standardize column names for easier processing
    df['Predicted_Label_Std'] = df[column_map['predicted']]
    df['True_Label_Std'] = df[column_map['true']]
    df['Confidence_Std'] = df[column_map['confidence']]
    
    # Create output directories
    categories = {
        'TN': ('True_Negative_Unbarred_Unbarred', 0, 0),  # True=0, Pred=0
        'FP': ('False_Positive_Unbarred_Barred', 0, 1),    # True=0, Pred=1
        'FN': ('False_Negative_Barred_Unbarred', 1, 0),    # True=1, Pred=0
        'TP': ('True_Positive_Barred_Barred', 1, 1)        # True=1, Pred=1
    }
    
    # Create base output directory
    base_path = Path(OUTPUT_BASE)
    base_path.mkdir(exist_ok=True)
    
    for cat_key, (folder_name, true_val, pred_val) in categories.items():
        cat_path = base_path / folder_name
        cat_path.mkdir(exist_ok=True)
        
        # Filter galaxies for this category
        mask = (df['True_Label_Std'] == true_val) & (df['Predicted_Label_Std'] == pred_val)
        category_galaxies = df[mask].copy()
        
        print(f"\n{'='*60}")
        print(f"Processing: {folder_name}")
        print(f"Count: {len(category_galaxies)} galaxies")
        print(f"{'='*60}")
        
        if len(category_galaxies) == 0:
            print(f"No galaxies in this category. Skipping...")
            continue
        
        # Collect galaxy data
        galaxy_data_batch = []
        batch_counter = 0
        
        for idx, row in category_galaxies.iterrows():
            galaxy_name = row['Target_Name']
            fits_path = find_fits_file(galaxy_name, FITS_FOLDER)
            
            if fits_path:
                print(f"  Found: {galaxy_name} -> {Path(fits_path).name}")
                data = load_fits_image(fits_path)
                galaxy_data_batch.append((
                    galaxy_name,
                    data,
                    row['Predicted_Label_Std'],
                    row['True_Label_Std'],
                    row['Confidence_Std']
                ))
            else:
                print(f"  Missing: {galaxy_name}")
                galaxy_data_batch.append((
                    galaxy_name,
                    None,
                    row['Predicted_Label_Std'],
                    row['True_Label_Std'],
                    row['Confidence_Std']
                ))
            
            # Create image when we have 4 galaxies
            if len(galaxy_data_batch) == GALAXIES_PER_IMAGE:
                batch_counter += 1
                output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
                create_galaxy_grid(galaxy_data_batch, output_file, folder_name.replace('_', ' '))
                galaxy_data_batch = []
        
        # Handle remaining galaxies
        if galaxy_data_batch:
            batch_counter += 1
            output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
            create_galaxy_grid(galaxy_data_batch, output_file, folder_name.replace('_', ' '))
    
    print("\n" + "="*60)
    print("Processing Complete!")
    print("="*60)
    print(f"\nOutput directory: {OUTPUT_BASE}/")
    print("\nFolder structure:")
    for cat_key, (folder_name, _, _) in categories.items():
        cat_path = base_path / folder_name
        if cat_path.exists():
            num_images = len(list(cat_path.glob('*.png')))
            print(f"  {folder_name}/: {num_images} images")

if __name__ == "__main__":
    process_galaxies()

Loading predictions CSV...
Loaded 117 galaxies

Detected CSV format: rf_predictions.csv format
Column mappings:
  - Predicted Label: RF_Predicted_Label
  - True Label: True_Label
  - Confidence: RF_Confidence

Processing: True_Negative_Unbarred_Unbarred
Count: 21 galaxies
  Found: ESO 034-G013 -> norm_resized_ESO_034-G013.fits
  Found: ESO 469-G015 -> norm_resized_ESO_469-G015.fits
  Found: IC 1071 -> norm_resized_IC_1071.fits
  Found: IC 1698 -> norm_resized_IC_1698.fits
Saved: confusion_matrix_galaxies_rf1\True_Negative_Unbarred_Unbarred\TN_batch_001.png
  Found: IC 4218 -> norm_resized_IC_4218.fits
  Found: IC 5264 -> norm_resized_IC_5264.fits
  Found: MESSIER 031 -> norm_resized_MESSIER_031.fits
  Found: NGC 0895 -> norm_resized_NGC_0895.fits
Saved: confusion_matrix_galaxies_rf1\True_Negative_Unbarred_Unbarred\TN_batch_002.png
  Found: NGC 1380A -> norm_resized_NGC_1380A.fits
  Found: NGC 4797 -> norm_resized_NGC_4797.fits
  Found: NGC 5577 -> norm_resized_NGC_5577.fits
  Found: NG

In [3]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix

# ============================================================================
# CONFIGURATION
# ============================================================================

MODEL_PATH = 'efficient_ring_cnn_final.h5'  # or 'best_efficient_model.h5'
TEST_IMAGE_FOLDER = 'test'
TEST_LABELS_CSV = 'testlabel.csv'
OUTPUT_CSV = 'cnn_predictions.csv'
OUTPUT_BASE = 'cnn_confusion_matrix_galaxies'
GALAXIES_PER_IMAGE = 4
IMAGE_SIZE = 224

# ============================================================================
# CUSTOM LAYER (needed for loading the model)
# ============================================================================

class RingFeatureLayer(keras.layers.Layer):
    """Custom layer for ring feature extraction"""
    def __init__(self, n_rings=8, **kwargs):
        super(RingFeatureLayer, self).__init__(**kwargs)
        self.n_rings = n_rings
    
    def build(self, input_shape):
        self.ring_weights = self.add_weight(
            name='ring_weights',
            shape=(self.n_rings,),
            initializer='ones',
            trainable=True
        )
        super(RingFeatureLayer, self).build(input_shape)
    
    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        
        cy = height // 2
        cx = width // 2
        
        y = tf.range(height, dtype=tf.float32)
        x = tf.range(width, dtype=tf.float32)
        yy, xx = tf.meshgrid(y, x, indexing='ij')
        
        dist = tf.maximum(tf.abs(xx - tf.cast(cx, tf.float32)), 
                         tf.abs(yy - tf.cast(cy, tf.float32)))
        
        max_dist = tf.cast(tf.minimum(cx, cy), tf.float32)
        ring_width = max_dist / tf.cast(self.n_rings, tf.float32)
        
        ring_features = []
        for i in range(self.n_rings):
            inner = tf.cast(i, tf.float32) * ring_width
            outer = tf.cast(i + 1, tf.float32) * ring_width
            
            mask = tf.cast((dist >= inner) & (dist < outer), tf.float32)
            mask = tf.reshape(mask, [1, height, width, 1])
            
            masked = inputs * mask
            
            ring_sum = tf.reduce_sum(masked, axis=[1, 2], keepdims=True)
            ring_count = tf.reduce_sum(mask, axis=[1, 2], keepdims=True) + 1e-6
            ring_mean = ring_sum / ring_count
            
            weighted_mean = ring_mean * self.ring_weights[i]
            ring_features.append(weighted_mean)
        
        ring_features = tf.concat(ring_features, axis=-1)
        ring_features = tf.squeeze(ring_features, axis=[1, 2])
        
        return ring_features
    
    def get_config(self):
        config = super(RingFeatureLayer, self).get_config()
        config.update({'n_rings': self.n_rings})
        return config

# ============================================================================
# DATA LOADING
# ============================================================================

def convert_target_name_to_filename(target_name):
    """Convert target name to FITS filename format"""
    fname = target_name.replace(' ', '_')
    return f"norm_resized_{fname}.fits"

def load_test_data(image_folder, labels_csv):
    """Load test data for prediction"""
    print(f"\nLoading test data from {image_folder}...")
    
    labels_df = pd.read_csv(labels_csv)
    labels_df = labels_df.dropna(subset=['Label'])
    
    images = []
    labels = []
    filenames = []
    
    for idx, row in labels_df.iterrows():
        target_name = row['Target Name']
        label = int(row['Label'])
        
        fname_actual = convert_target_name_to_filename(target_name)
        fpath = os.path.join(image_folder, fname_actual)
        
        if not os.path.exists(fpath):
            print(f"  Warning: File not found - {fname_actual}")
            continue
        
        try:
            with fits.open(fpath) as hdul:
                data = np.squeeze(hdul[0].data)
                
                if data.ndim != 2:
                    print(f"  Warning: Skipping {fname_actual} - not 2D")
                    continue
                
                images.append(data)
                labels.append(label)
                filenames.append(target_name)
                
        except Exception as e:
            print(f"  Error loading {fname_actual}: {e}")
            continue
    
    images = np.array(images)
    labels = np.array(labels)
    
    # Add channel dimension
    if images.ndim == 3:
        images = np.expand_dims(images, axis=-1)
    
    print(f"✓ Loaded {len(images)} test images")
    print(f"  Unbarred (0): {np.sum(labels==0)}, Barred (1): {np.sum(labels==1)}")
    
    return images, labels, filenames

# ============================================================================
# PREDICTION
# ============================================================================

def generate_predictions(model, X_test, y_test, filenames):
    """Generate predictions and save to CSV"""
    print("\n" + "="*70)
    print("GENERATING PREDICTIONS")
    print("="*70)
    
    # Get predictions
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    
    # Create results dataframe
    results = pd.DataFrame({
        'Target_Name': filenames,
        'True_Label': y_test,
        'Predicted_Label': y_pred,
        'Confidence': y_pred_proba.flatten(),
        'Correct': y_pred == y_test
    })
    
    # Save to CSV
    results.to_csv(OUTPUT_CSV, index=False)
    print(f"✓ Predictions saved to '{OUTPUT_CSV}'")
    
    # Print summary
    accuracy = np.mean(y_pred == y_test)
    print(f"\n✓ Overall Accuracy: {accuracy:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"              Predicted")
    print(f"            Unbarred  Barred")
    print(f"  Actual")
    print(f"  Unbarred    {cm[0,0]:4d}    {cm[0,1]:4d}")
    print(f"  Barred      {cm[1,0]:4d}    {cm[1,1]:4d}")
    
    # Category breakdown
    tn = cm[0, 0]  # True Negative
    fp = cm[0, 1]  # False Positive
    fn = cm[1, 0]  # False Negative
    tp = cm[1, 1]  # True Positive
    
    print(f"\nCategory counts:")
    print(f"  True Negative (TN): {tn} - Correctly predicted Unbarred")
    print(f"  False Positive (FP): {fp} - Unbarred predicted as Barred")
    print(f"  False Negative (FN): {fn} - Barred predicted as Unbarred")
    print(f"  True Positive (TP): {tp} - Correctly predicted Barred")
    
    return results

# ============================================================================
# VISUALIZATION
# ============================================================================

def normalize_galaxy_name(name):
    """Convert galaxy name to match FITS filename format"""
    return name.replace(' ', '_')

def find_fits_file(galaxy_name, fits_folder):
    """Find the corresponding FITS file for a galaxy"""
    normalized_name = normalize_galaxy_name(galaxy_name)
    
    patterns = [
        f"norm_resized_{normalized_name}.fits",
        f"norm_resized_{normalized_name.replace('_', '-')}.fits",
        f"norm_resized_{normalized_name.replace('-', '_')}.fits",
    ]
    
    for pattern in patterns:
        filepath = Path(fits_folder) / pattern
        if filepath.exists():
            return str(filepath)
    
    return None

def load_fits_image(fits_path):
    """Load and return image data from FITS file"""
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data
            if data.ndim > 2:
                data = data[0] if data.shape[0] < data.shape[-1] else data
            return data
    except Exception as e:
        print(f"Error loading {fits_path}: {e}")
        return None

def create_galaxy_grid(galaxy_data_list, output_path, category_name):
    """Create a 2x2 grid of galaxy images"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(category_name, fontsize=16, fontweight='bold')
    
    axes = axes.flatten()
    
    for idx, (name, data, pred_label, true_label, prob) in enumerate(galaxy_data_list):
        if idx >= 4:
            break
            
        ax = axes[idx]
        
        if data is not None:
            vmin, vmax = np.percentile(data, [1, 99])
            im = ax.imshow(data, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
            
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
            pred_str = "Barred" if pred_label == 1 else "Unbarred"
            true_str = "Barred" if true_label == 1 else "Unbarred"
            title = f"{name}\nTrue: {true_str}, Pred: {pred_str}\nConf: {prob:.3f}"
            ax.set_title(title, fontsize=10, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'{name}\nNot Found', 
                   ha='center', va='center', fontsize=12)
            ax.set_title(f"{name} (Missing)", fontsize=10)
        
        ax.axis('off')
    
    for idx in range(len(galaxy_data_list), 4):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

def visualize_confusion_matrix(predictions_df, fits_folder):
    """Create confusion matrix visualizations"""
    print("\n" + "="*70)
    print("CREATING CONFUSION MATRIX VISUALIZATIONS")
    print("="*70)
    
    categories = {
        'TN': ('True_Negative_Unbarred_Unbarred', 0, 0),
        'FP': ('False_Positive_Unbarred_Barred', 0, 1),
        'FN': ('False_Negative_Barred_Unbarred', 1, 0),
        'TP': ('True_Positive_Barred_Barred', 1, 1)
    }
    
    base_path = Path(OUTPUT_BASE)
    base_path.mkdir(exist_ok=True)
    
    for cat_key, (folder_name, true_val, pred_val) in categories.items():
        cat_path = base_path / folder_name
        cat_path.mkdir(exist_ok=True)
        
        mask = (predictions_df['True_Label'] == true_val) & \
               (predictions_df['Predicted_Label'] == pred_val)
        category_galaxies = predictions_df[mask].copy()
        
        print(f"\n{'='*60}")
        print(f"Processing: {folder_name}")
        print(f"Count: {len(category_galaxies)} galaxies")
        print(f"{'='*60}")
        
        if len(category_galaxies) == 0:
            print(f"No galaxies in this category. Skipping...")
            continue
        
        galaxy_data_batch = []
        batch_counter = 0
        
        for idx, row in category_galaxies.iterrows():
            galaxy_name = row['Target_Name']
            fits_path = find_fits_file(galaxy_name, fits_folder)
            
            if fits_path:
                print(f"  Found: {galaxy_name} -> {Path(fits_path).name}")
                data = load_fits_image(fits_path)
                galaxy_data_batch.append((
                    galaxy_name,
                    data,
                    row['Predicted_Label'],
                    row['True_Label'],
                    row['Confidence']
                ))
            else:
                print(f"  Missing: {galaxy_name}")
                galaxy_data_batch.append((
                    galaxy_name,
                    None,
                    row['Predicted_Label'],
                    row['True_Label'],
                    row['Confidence']
                ))
            
            if len(galaxy_data_batch) == GALAXIES_PER_IMAGE:
                batch_counter += 1
                output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
                create_galaxy_grid(galaxy_data_batch, output_file, 
                                 folder_name.replace('_', ' '))
                galaxy_data_batch = []
        
        if galaxy_data_batch:
            batch_counter += 1
            output_file = cat_path / f"{cat_key}_batch_{batch_counter:03d}.png"
            create_galaxy_grid(galaxy_data_batch, output_file, 
                             folder_name.replace('_', ' '))
    
    print("\n" + "="*60)
    print("Processing Complete!")
    print("="*60)
    print(f"\nOutput directory: {OUTPUT_BASE}/")
    print("\nFolder structure:")
    for cat_key, (folder_name, _, _) in categories.items():
        cat_path = base_path / folder_name
        if cat_path.exists():
            num_images = len(list(cat_path.glob('*.png')))
            print(f"  {folder_name}/: {num_images} images")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    print("\n" + "="*70)
    print("CNN CONFUSION MATRIX GENERATOR")
    print("="*70)
    print("This script will:")
    print("  1. Load your trained CNN model")
    print("  2. Generate predictions on test data")
    print("  3. Save predictions to CSV")
    print("  4. Create confusion matrix visualizations")
    print("="*70)
    
    # Check if model exists
    if not os.path.exists(MODEL_PATH):
        print(f"\n✗ ERROR: Model file '{MODEL_PATH}' not found!")
        print("Please check the MODEL_PATH in the configuration.")
        exit(1)
    
    # Load model with custom objects
    print(f"\nLoading model from '{MODEL_PATH}'...")
    try:
        model = keras.models.load_model(
            MODEL_PATH,
            custom_objects={'RingFeatureLayer': RingFeatureLayer}
        )
        print("✓ Model loaded successfully")
    except Exception as e:
        print(f"✗ Error loading model: {e}")
        exit(1)
    
    # Load test data
    X_test, y_test, filenames = load_test_data(TEST_IMAGE_FOLDER, TEST_LABELS_CSV)
    
    if len(X_test) == 0:
        print("\n✗ ERROR: No test data loaded!")
        exit(1)
    
    # Generate predictions
    predictions_df = generate_predictions(model, X_test, y_test, filenames)
    
    # Create visualizations
    visualize_confusion_matrix(predictions_df, TEST_IMAGE_FOLDER)
    
    print("\n" + "="*70)
    print("✓ ALL DONE!")
    print("="*70)
    print(f"✓ Predictions CSV: {OUTPUT_CSV}")
    print(f"✓ Visualizations: {OUTPUT_BASE}/")
    print("="*70)



CNN CONFUSION MATRIX GENERATOR
This script will:
  1. Load your trained CNN model
  2. Generate predictions on test data
  3. Save predictions to CSV
  4. Create confusion matrix visualizations

Loading model from 'efficient_ring_cnn_final.h5'...


✓ Model loaded successfully

Loading test data from test...
✓ Loaded 117 test images
  Unbarred (0): 48, Barred (1): 69

GENERATING PREDICTIONS
✓ Predictions saved to 'cnn_predictions.csv'

✓ Overall Accuracy: 0.6667

Confusion Matrix:
              Predicted
            Unbarred  Barred
  Actual
  Unbarred       9      39
  Barred         0      69

Category counts:
  True Negative (TN): 9 - Correctly predicted Unbarred
  False Positive (FP): 39 - Unbarred predicted as Barred
  False Negative (FN): 0 - Barred predicted as Unbarred
  True Positive (TP): 69 - Correctly predicted Barred

CREATING CONFUSION MATRIX VISUALIZATIONS

Processing: True_Negative_Unbarred_Unbarred
Count: 9 galaxies
  Found: IC 1698 -> norm_resized_IC_1698.fits
  Foun